# AI suggesting detection rules

# Coding problem, 1.5.c: Generate detection rules from a login log

**Skill:** Detect Attacks · **Maps to:** 1.5.A · **Language:** Python 3 (pytest)

## Problem
A SOC receives a stream of authentication log events. Your task is to implement a
brute-force detection engine: scan the log for source IPs that generate 5 or more
`login_fail` events within any 60-second window, and emit a structured detection-rule
string for each flagged IP.

This mirrors how an AI-assisted SIEM generates candidate detection rules from raw log data.

## Function contract
Implement `solve(logs: list[dict]) -> list[str]` in the code editor
(see the Reference section below).

## Input
`logs`, a list of dicts, each with:
- `ts` (int): Unix timestamp in seconds
- `src` (str): source IP address
- `event` (str): event type; only `"login_fail"` counts toward brute-force detection
- `ok` (bool): True = success, False = failure

## Output
A **sorted** list of detection-rule strings. Format for each triggered IP:

```
ALERT brute_force src=<IP> count=<N> window=60s
```

where `<N>` is the maximum number of `login_fail` events from that IP found within any
contiguous 60-second window (i.e., events where `max_ts - min_ts <= 59`).
Rules are sorted lexicographically. At most one rule per IP.

## Examples
```python
logs = [
    {"ts": 0,  "src": "1.2.3.4", "event": "login_fail", "ok": False},
    {"ts": 10, "src": "1.2.3.4", "event": "login_fail", "ok": False},
    {"ts": 20, "src": "1.2.3.4", "event": "login_fail", "ok": False},
    {"ts": 30, "src": "1.2.3.4", "event": "login_fail", "ok": False},
    {"ts": 40, "src": "1.2.3.4", "event": "login_fail", "ok": False},
]
solve(logs) -> ["ALERT brute_force src=1.2.3.4 count=5 window=60s"]

logs = [
    {"ts": 0,  "src": "9.9.9.9", "event": "login_fail", "ok": False},
    {"ts": 10, "src": "9.9.9.9", "event": "login_fail", "ok": False},
]
solve(logs) -> []   # only 2 failures — below threshold

solve([]) -> []     # empty log
```

## Algorithm (sliding window)
1. Collect all `login_fail` events and group by `src`.
2. Sort each group by `ts`.
3. For each group, slide a window: for each event at position `i`, count how many
   events starting at `i` have `ts <= events[i].ts + 59`.
4. If the maximum window count for any source is >= 5, emit its rule string.
5. Return the list sorted lexicographically.

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(logs: list[dict]) -> list[str]
```

### Input (`logs`)
A list of log-event dicts. Each dict has exactly these keys:

| Key    | Type | Description                                   |
|--------|------|-----------------------------------------------|
| `ts`   | int  | Unix timestamp (seconds, non-negative integer) |
| `src`  | str  | Source IP address string (e.g. `"10.0.0.1"`)  |
| `event`| str  | Event type string; `"login_fail"` is the only type that counts toward the brute-force rule |
| `ok`   | bool | `True` = success, `False` = failure (redundant with event, included for completeness) |

Example single entry:
```python
{"ts": 1000, "src": "10.0.0.5", "event": "login_fail", "ok": False}
```

### Output (`result`)
A **sorted** `list[str]` of detection-rule strings. One rule string is emitted per source IP
that triggers the brute-force condition. The exact rule string format is:

```
ALERT brute_force src=<IP> count=<N> window=60s
```

Where:
- `<IP>` is the source IP string exactly as it appears in the log entries.
- `<N>` is the integer count of `login_fail` events from that IP within any contiguous 60-second
  window (the maximum count observed across all possible 60-second windows for that IP).
- `window=60s` is the literal string (fixed).

The list is sorted lexicographically (Python default `str` sort on the full rule string).

### Brute-force detection rule
An IP triggers the rule when it produces **5 or more** `login_fail` events where all events fall
within a 60-second window (i.e., `max_ts - min_ts <= 59` for some subset of size >= 5 from that
IP's `login_fail` events).

Detection algorithm (reference implementation uses this):
1. Group all `login_fail` events by `src`.
2. For each source, sort events by `ts`.
3. Use a sliding window: for each event at index `i`, count how many subsequent events have
   `ts <= events[i].ts + 59` (inclusive). The window size is the length of that group.
4. If the maximum window size for that source is >= 5, emit the rule with `count=` set to that
   maximum window size.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- `logs` is a list (may be empty); entries are dicts with the schema above.
- `ts` values are non-negative integers; multiple events can share the same `ts`.
- Other event types (e.g. `"login_ok"`, `"logout"`) are ignored.
- Output is always sorted ascending (lexicographic) and de-duplicated (at most one rule per IP).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.5.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List, Dict, Any


def solve(logs: List[Dict[str, Any]]) -> List[str]:
    """Generate detection-rule strings for IPs that exceed the brute-force threshold.

    Args:
        logs: list of dicts with keys ts (int), src (str), event (str), ok (bool).
              Only 'login_fail' events count toward the brute-force rule.
    Returns:
        Sorted list of rule strings of the form:
        'ALERT brute_force src=<IP> count=<N> window=60s'
        One entry per IP that has >= 5 login_fail events within any 60-second window.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() -- see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _fail(src, ts_list):
    """Helper: build login_fail events for src at given timestamps."""
    return [{"ts": t, "src": src, "event": "login_fail", "ok": False} for t in ts_list]


def _ok(src, ts):
    return {"ts": ts, "src": src, "event": "login_ok", "ok": True}


# ---- visible cases -------------------------------------------------------

def test_single_ip_exactly_threshold():
    logs = _fail("10.0.0.1", [0, 10, 20, 30, 40])
    result = impl.solve(logs)
    assert result == ["ALERT brute_force src=10.0.0.1 count=5 window=60s"]


def test_below_threshold_not_flagged():
    logs = _fail("10.0.0.2", [0, 30, 59])
    assert impl.solve(logs) == []


def test_empty_logs():
    assert impl.solve([]) == []


def test_two_ips_one_flagged():
    logs = _fail("192.168.1.1", [0, 5, 10, 15, 20]) + _fail("192.168.1.2", [0, 100])
    result = impl.solve(logs)
    assert result == ["ALERT brute_force src=192.168.1.1 count=5 window=60s"]


def test_login_ok_events_ignored():
    # 4 failures + many successes = should NOT trigger
    logs = _fail("172.16.0.1", [0, 10, 20, 30]) + [_ok("172.16.0.1", t) for t in range(100)]
    assert impl.solve(logs) == []


# ---- hidden / edge cases -------------------------------------------------

def test_window_boundary_inclusive():
    # Events at ts=0 and ts=59 are exactly 59 apart -- same 60-second window
    logs = _fail("10.1.1.1", [0, 10, 20, 30, 59])
    result = impl.solve(logs)
    assert result == ["ALERT brute_force src=10.1.1.1 count=5 window=60s"]


def test_window_boundary_exclusive():
    # Events at ts=0 and ts=60 are 60 apart -- outside the 60-second window
    # Only events 0..4 fit in window starting at 0 if last is ts=60
    # But only events ts=1..59 fit for window starting 1, etc.
    # With ts=[0,15,30,45,60]: window [0..59] covers [0,15,30,45] -> 4 only
    logs = _fail("10.1.1.2", [0, 15, 30, 45, 60])
    assert impl.solve(logs) == []


def test_two_ips_both_flagged_sorted():
    logs = _fail("203.0.113.5", [0, 1, 2, 3, 4]) + _fail("198.51.100.1", [100, 101, 102, 103, 104])
    result = impl.solve(logs)
    assert result == sorted(result)
    assert len(result) == 2
    assert "ALERT brute_force src=198.51.100.1 count=5 window=60s" in result
    assert "ALERT brute_force src=203.0.113.5 count=5 window=60s" in result


def test_count_reflects_max_window():
    # 7 events within 60s; count should be 7 not 5
    logs = _fail("10.2.2.2", [0, 5, 10, 15, 20, 25, 30])
    result = impl.solve(logs)
    assert result == ["ALERT brute_force src=10.2.2.2 count=7 window=60s"]


def test_events_spread_across_two_windows():
    # 3 in first window, 5 in second; only second triggers
    logs = _fail("10.3.3.3", [0, 20, 40, 100, 110, 120, 130, 140])
    result = impl.solve(logs)
    assert result == ["ALERT brute_force src=10.3.3.3 count=5 window=60s"]


def test_no_login_fail_events():
    logs = [_ok("10.4.4.4", t) for t in range(20)]
    assert impl.solve(logs) == []


def test_duplicate_timestamps():
    # 5 events at same timestamp -- all within 60s window
    logs = _fail("10.5.5.5", [50, 50, 50, 50, 50])
    result = impl.solve(logs)
    assert result == ["ALERT brute_force src=10.5.5.5 count=5 window=60s"]


def test_output_is_sorted():
    # Multiple IPs -- output must be lexicographically sorted
    ips = ["99.1.1.1", "10.0.0.1", "172.16.0.1"]
    logs = []
    for ip in ips:
        logs += _fail(ip, [0, 1, 2, 3, 4])
    result = impl.solve(logs)
    assert result == sorted(result)
    assert len(result) == 3


def test_exactly_four_failures_not_flagged():
    logs = _fail("10.6.6.6", [0, 10, 20, 30])
    assert impl.solve(logs) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
